# Farm Agent — 3. Vision: reproducible split, retrain YOLOv8s-cls, held-out test metrics

**Settings:** Accelerator = GPU T4 · Internet = On. ~20–30 min.
**Inputs to attach:** `emmarex/plantdisease`, `vencerlanz09/agricultural-pests-image-dataset` (and the repo dataset, optional).

The original `best.pt` reported 93.8% top-1 on its *validation* split, but that split used an unseeded shuffle,
so its test images can't be reconstructed. This notebook retrains on a deterministic split and reports
test accuracy, macro-F1, per-class F1, calibration (ECE) and coverage at the app's confidence gates. The model this produced (96.5% test top-1) is the one now shipped as `outputs/farm-vision/weights/best.pt`.

Work happens in `/tmp`; only results land in `/kaggle/working` (so the notebook output stays small).

In [ ]:
REPO = "https://github.com/shashank-sj/farm-agent.git"
BRANCH = "main"
# Code source: a private Kaggle dataset with the repo (attached as input) if present, else GitHub.
import glob, os, shutil, tarfile, zipfile
DST = "/tmp/farm-agent"
shutil.rmtree(DST, ignore_errors=True)
apps = glob.glob("/kaggle/input/**/app.py", recursive=True)
archives = glob.glob("/kaggle/input/**/farm_agent_src.*", recursive=True)
if apps:
    shutil.copytree(os.path.dirname(apps[0]), DST)
elif archives:
    os.makedirs(DST)
    a = archives[0]
    (tarfile.open(a) if a.endswith((".tar.gz", ".tgz")) else zipfile.ZipFile(a)).extractall(DST)
else:
    !git clone --depth 1 -b $BRANCH $REPO $DST
%cd $DST

def find_input(slug, handle):
    """Directory of an attached Kaggle dataset (any mount layout); download only as a fallback."""
    for d in glob.glob("/kaggle/input/**/", recursive=True):
        if os.path.basename(os.path.normpath(d)) == slug:
            return d
    import kagglehub
    return kagglehub.dataset_download(handle)

!pip install -q ultralytics scikit-learn

In [ ]:
PV = find_input("plantdisease", "emmarex/plantdisease")
PESTS = find_input("agricultural-pests-image-dataset", "vencerlanz09/agricultural-pests-image-dataset")
print(PV, PESTS)
!python -m scripts.train_vision --plantvillage "$PV" --pests "$PESTS" --work /tmp/farm-vision-dataset --out outputs/farm-vision-v2

In [ ]:
OUT = "/kaggle/working"
!mkdir -p $OUT/vision
!cp outputs/farm-vision-v2/test_metrics.json outputs/farm-vision-v2/confusion_matrix.csv $OUT/vision/
!cp outputs/farm-vision-v2/weights/best.pt $OUT/vision/best_v2.pt
!cp outputs/farm-vision-v2/train/results.csv $OUT/vision/train_results.csv 2>/dev/null; true
!ls -la $OUT/vision